# Izziv: Analiza besedila o podatkovni znanosti

V tem primeru izvedimo preprosto vajo, ki zajema vse korake tradicionalnega procesa podatkovne znanosti. Ni vam treba pisati nobene kode, lahko samo kliknete na spodnje celice, da jih zaženete in opazujete rezultat. Kot izziv ste vabljeni, da to kodo preizkusite z različnimi podatki.

## Cilj

V tej lekciji smo razpravljali o različnih konceptih, povezanih s podatkovno znanostjo. Poskusimo odkriti več sorodnih konceptov z **rudarenjem besedila**. Začeli bomo z besedilom o podatkovni znanosti, iz njega izvlekli ključne besede in nato poskusili vizualizirati rezultat.

Kot besedilo bom uporabil stran o podatkovni znanosti iz Wikipedije:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## 1. korak: Pridobivanje podatkov

Prvi korak v vsakem procesu podatkovne znanosti je pridobivanje podatkov. Za to bomo uporabili knjižnico `requests`:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Korak 2: Pretvorba podatkov

Naslednji korak je, da podatke pretvorimo v obliko, primerno za obdelavo. V našem primeru smo prenesli izvorno kodo HTML s strani in jo moramo pretvoriti v navadno besedilo.

Obstaja veliko načinov, kako to narediti. Uporabili bomo [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), priljubljeno Python knjižnico za razčlenjevanje HTML. BeautifulSoup nam omogoča ciljanje določenih HTML elementov, tako da se lahko osredotočimo na glavno vsebino članka na Wikipediji in zmanjšamo nekaj menijev za navigacijo, stranskih vrstic, nog in druge nepomembne vsebine (čeprav lahko nekaj splošnega besedila še ostane).


Najprej moramo namestiti knjižnico BeautifulSoup za razčlenjevanje HTML: 


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Korak 3: Pridobivanje vpogledov

Najpomembnejši korak je, da naše podatke pretvorimo v neko obliko, iz katere lahko potegnemo vpoglede. V našem primeru želimo iz besedila izvleči ključne besede in videti, katere ključne besede so bolj pomenljive.

Uporabili bomo Python knjižnico z imenom [RAKE](https://github.com/aneesha/RAKE) za izvleček ključnih besed. Najprej pa namestimo to knjižnico, če ni že prisotna:


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Glavna funkcionalnost je na voljo v objektu `Rake`, ki ga lahko prilagodimo z nekaterimi parametri. V našem primeru bomo nastavili minimalno dolžino ključne besede na 5 znakov, minimalno frekvenco ključne besede v dokumentu na 3 in največje število besed v ključni besedi - na 2. Prosto eksperimentirajte z drugimi vrednostmi in opazujte rezultat.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Pridobili smo seznam izrazov skupaj s pripadajočo stopnjo pomembnosti. Kot lahko vidite, so najpomembnejše discipline, kot sta strojno učenje in veliki podatki, prisotne na vrhu seznama.

## 4. korak: Vizualizacija rezultata

Ljudje lahko podatke najbolje interpretirajo v vizualni obliki. Zato je pogosto smiselno podatke vizualizirati, da bi izluščili določene vpoglede. Za preprosto prikazovanje porazdelitve ključnih besed z njihovo pomembnostjo lahko uporabimo knjižnico `matplotlib` v Pythonu:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Obstaja pa še boljši način za vizualizacijo pogostosti besed – z uporabo **Word Cloud**. Za risanje oblačka besed iz našega seznama ključnih besed bomo morali namestiti še eno knjižnico.


In [ ]:
!{sys.executable} -m pip install wordcloud

Objekt `WordCloud` je odgovoren za sprejem originalnega besedila ali predhodno izračunanega seznama besed z njihovimi frekvencami ter vrne sliko, ki jo je mogoče nato prikazati z uporabo `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Izvirno besedilo lahko tudi posredujemo `WordCloud` - poglejmo, ali bomo lahko dobili podoben rezultat:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Vidite lahko, da oblak besed zdaj izgleda bolj impresivno, vendar vsebuje tudi veliko šuma (npr. nepovezane besede, kot je `Retrieved on`). Prav tako dobimo manj ključnih besed, ki jih sestavljata dve besedi, kot so *data scientist* ali *computer science*. To je zato, ker RAKE algoritem precej bolje opravi delo pri izbiri dobrih ključnih besed iz besedila. Ta primer prikazuje pomen predobdelave in čiščenja podatkov, saj nam bo jasna slika na koncu omogočila boljše odločitve.

V tej vaji smo šli skozi preprost postopek izvlečenja pomena iz besedila Wikipedije, v obliki ključnih besed in oblaka besed. Ta primer je precej preprost, vendar dobro prikaže vse tipične korake, ki jih bo podatkovni znanstvenik naredil pri delu s podatki, od pridobivanja podatkov do vizualizacije.

V našem tečaju bomo vse te korake podrobno obravnavali. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
